In [35]:
import pandas as pd
import numpy as np

In [36]:
# =====================================================
# Load Dataset & Basic Preprocessing
# =====================================================

df = pd.read_csv('../Data/electricity_demand.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.set_index('datetime')

In [37]:
# ===== LAG FEATURES =====
df['lag_1']   = df['demand_MW'].shift(1)   
df['lag_24']  = df['demand_MW'].shift(24)
df['lag_168'] = df['demand_MW'].shift(168) 

In [38]:
# ===== ROLLING FEATURES =====
df['rolling_24_mean'] = df['demand_MW'].shift(1).rolling(24).mean()
df['rolling_24_std'] = df['demand_MW'].shift(1).rolling(24).std()

df['rolling_168_mean'] = df['demand_MW'].shift(1).rolling(168).mean()
df['rolling_168_std'] = df['demand_MW'].shift(1).rolling(168).std()

In [39]:
# ===== EXPANDING FEATURES =====
df['expanding_sum'] = df['demand_MW'].expanding().sum()
df['expanding_mean'] = df['demand_MW'].expanding().mean()
df['expanding_std'] = df['demand_MW'].expanding().std()

In [40]:
print(f"Total features created: {len(df.columns)}") 
print("\nFeature columns:")
print(df.columns.tolist()) 

Total features created: 22

Feature columns:
['demand_MW', 'temperature', 'humidity', 'wind_speed', 'pressure', 'precipitation', 'holiday', 'weekend', 'hour', 'day_of_week', 'month', 'season', 'lag_1', 'lag_24', 'lag_168', 'rolling_24_mean', 'rolling_24_std', 'rolling_168_mean', 'rolling_168_std', 'expanding_sum', 'expanding_mean', 'expanding_std']


In [41]:
nan_count = df.isnull().sum()
print(nan_count[nan_count > 0])

lag_1                 1
lag_24               24
lag_168             168
rolling_24_mean      24
rolling_24_std       24
rolling_168_mean    168
rolling_168_std     168
expanding_std         1
dtype: int64


In [42]:
print(f"\nOriginal rows : {len(df)}")

df = df.dropna()
print(f"After dropna  : {len(df)}")
print(f"Rows removed  : {len(df) - len(df)}")
print(f"Data loss %   : {((len(df) - len(df)) / len(df)) * 100:.4f}%")


Original rows : 26280
After dropna  : 26112
Rows removed  : 0
Data loss %   : 0.0000%


In [43]:
corr = df.corr(numeric_only=True)
corr['demand_MW'].sort_values(ascending=False)

demand_MW           1.000000
lag_24              0.960063
lag_1               0.920999
lag_168             0.899520
rolling_24_mean     0.734371
rolling_168_mean    0.698224
hour                0.400366
rolling_24_std      0.280673
rolling_168_std     0.265452
temperature         0.235138
expanding_sum       0.139923
expanding_mean      0.094203
expanding_std      -0.002036
precipitation      -0.004962
pressure           -0.007213
wind_speed         -0.013946
month              -0.019530
holiday            -0.023331
humidity           -0.038054
day_of_week        -0.087931
weekend            -0.117431
Name: demand_MW, dtype: float64

In [44]:
features = [
    # Lag (top 3 strongest)
    'lag_1',
    'lag_24',
    'lag_168',

    # Rolling
    'rolling_24_mean',
    'rolling_168_mean',

    # Time features
    'hour',          # 0.40 - strong
    'day_of_week',   # -0.08 - meaningful
    'weekend',       # -0.11 - meaningful
    'holiday',       

    # Weather
    'temperature',   # 0.23 - useful
]

In [45]:
cols_to_save = features + ['demand_MW']

df_save = df[cols_to_save].copy()

# Save
df_save.to_csv('feature-columns.csv', index=True)  

print(f"✅ Saved!")
print(f"Shape: {df_save.shape}")
print(df_save.head())

✅ Saved!
Shape: (26112, 11)
                      lag_1  lag_24  lag_168  rolling_24_mean  \
datetime                                                        
2021-01-08 00:00:00  4626.0  3650.2   3496.9      4761.875000   
2021-01-08 01:00:00  3849.4  3735.9   3525.2      4770.175000   
2021-01-08 02:00:00  4002.9  3843.8   3290.1      4781.300000   
2021-01-08 03:00:00  3899.4  3717.4   3284.3      4783.616667   
2021-01-08 04:00:00  3634.3  3664.3   3449.4      4780.154167   

                     rolling_168_mean  hour  day_of_week  weekend  holiday  \
datetime                                                                     
2021-01-08 00:00:00       4603.267262     0            4        0        0   
2021-01-08 01:00:00       4605.365476     1            4        0        0   
2021-01-08 02:00:00       4608.208929     2            4        0        0   
2021-01-08 03:00:00       4611.835714     3            4        0        0   
2021-01-08 04:00:00       4613.919048     4     